# ACTO SDK: Registry Operations

This notebook demonstrates how to use the ProofRegistry with context managers and async operations.


In [ ]:
import asyncio
from datetime import datetime

from acto.crypto import KeyPair
from acto.proof import create_proof
from acto.registry import AsyncProofRegistry, ProofRegistry
from acto.telemetry.models import TelemetryBundle, TelemetryEvent


## Using Registry with Context Manager


In [ ]:
# Generate keypair and create proof
keypair = KeyPair.generate()
bundle = TelemetryBundle(
    task_id="registry-demo-001",
    robot_id="robot-001",
    events=[
        TelemetryEvent(
            ts=datetime.now().isoformat(),
            topic="test",
            data={"test": "data"}
        )
    ]
)

envelope = create_proof(
    bundle,
    keypair.private_key_b64,
    keypair.public_key_b64
)

# Use registry as context manager
with ProofRegistry() as registry:
    # Store proof
    proof_id = registry.upsert(envelope)
    print(f"Stored proof: {proof_id}")
    
    # Retrieve proof
    retrieved = registry.get(proof_id)
    print(f"Retrieved proof: {retrieved.payload.subject.task_id}")
    
    # List proofs
    proofs = registry.list(limit=10)
    print(f"Found {len(proofs)} proofs in registry")


## Async Registry Operations


In [ ]:
async def async_registry_example():
    # Use async registry as context manager
    async with AsyncProofRegistry() as registry:
        # Store proof
        proof_id = await registry.upsert(envelope)
        print(f"Stored proof: {proof_id}")
        
        # Retrieve proof
        retrieved = await registry.get(proof_id)
        print(f"Retrieved proof: {retrieved.payload.subject.task_id}")
        
        # List proofs
        proofs = await registry.list(limit=10)
        print(f"Found {len(proofs)} proofs")
        
        # Search proofs
        results = await registry.search("registry-demo", limit=5)
        print(f"Search results: {len(results)}")

await async_registry_example()


## Batch Operations


In [ ]:
async def batch_registry_operations():
    # Create multiple proofs
    envelopes = []
    for i in range(5):
        bundle = TelemetryBundle(
            task_id=f"batch-task-{i:03d}",
            robot_id="robot-001",
            events=[
                TelemetryEvent(
                    ts=datetime.now().isoformat(),
                    topic="sensor",
                    data={"index": i}
                )
            ]
        )
        envelope = create_proof(
            bundle,
            keypair.private_key_b64,
            keypair.public_key_b64
        )
        envelopes.append(envelope)
    
    # Store all proofs concurrently
    async with AsyncProofRegistry() as registry:
        tasks = [registry.upsert(env) for env in envelopes]
        proof_ids = await asyncio.gather(*tasks)
        print(f"Stored {len(proof_ids)} proofs")
        
        # Retrieve all proofs concurrently
        get_tasks = [registry.get(pid) for pid in proof_ids]
        retrieved = await asyncio.gather(*get_tasks)
        print(f"Retrieved {len(retrieved)} proofs")

await batch_registry_operations()
